# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w08_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Same window as ML-04/05/06/07/08 (w03-w07) — mid-panel month, never the sealed June 2026 sample.
MONTH_START = "2026-03-01"
MONTH_END_EXCL = "2026-04-01"      # half-open: report_date < MONTH_END_EXCL
PREV30_START = "2026-01-30"        # the 30 days immediately before MONTH_START
PREV30_END_EXCL = MONTH_START

print(f"Connected. Iterating on month={MONTH_START[:7]} | prev30 window: [{PREV30_START}, {PREV30_END_EXCL})")

# --- Rebuild the identical feature vector + label as ML-05/07/08 ---
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev30,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{PREV30_START}' AND report_date < DATE '{PREV30_END_EXCL}'
    GROUP BY 1, 2
""").df()
feature_frame["ctr_prev30"] = (feature_frame["clk_prev30"] / feature_frame["imp_prev30"]).fillna(0)

content_schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
EXCLUDE_LIKE = ("client", "hash", "id", "profile", "account", "flag", "score")
candidate_cols = [
    c for c in content_schema.loc[content_schema["column_type"] == "VARCHAR", "column_name"]
    if not any(bad in c.lower() for bad in EXCLUDE_LIKE)
]
cat_features = []
for col in candidate_cols:
    n_distinct = con.sql(f"SELECT COUNT(DISTINCT {col}) FROM {TABLES['dim_content']}").fetchone()[0]
    if 1 < n_distinct <= 15:
        cat_features.append(col)

cols_sql = ", ".join(cat_features)
content_meta = con.sql(f"SELECT content_hash_id, {cols_sql} FROM {TABLES['dim_content']}").df()
feature_frame = feature_frame.merge(content_meta, on="content_hash_id", how="left")
for col in cat_features:
    feature_frame[col] = feature_frame[col].fillna("unknown")
feature_frame = pd.get_dummies(feature_frame, columns=cat_features, prefix=cat_features)

march = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2
""").df()
data = feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["imp_prev30"] > 0].copy()
data["is_declining"] = (data["imp_march"] < 0.8 * data["imp_prev30"]).astype(int)

feature_cols = [c for c in feature_frame.columns if c not in ("client_hash_id", "content_hash_id")]
print(f"\n{len(data):,} content items, {len(feature_cols)} feature columns, base rate {data['is_declining'].mean():.3f}")
print("Same 146,253 rows / 0.285 base rate as ML-05/07/08 if the pipeline is consistent.")

Paste your Hugging Face READ token (hf_...): ··········
Connected. Iterating on month=2026-03 | prev30 window: [2026-01-30, 2026-03-01)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


146,253 content items, 22 feature columns, base rate 0.285
Same 146,253 rows / 0.285 base rate as ML-05/07/08 if the pipeline is consistent.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Scope note:** the paper (`docs/flyrank-seo-research-march-2026.pdf`) runs on a different
portfolio than mine — 57 brands / 341,701 content pieces, vs. my March-2026 slice's 321,546
items and 29/13-client grouped split. This section audits the paper's own internal validation
logic, not a re-derivation against my warehouse pull.

**Finding #4 — "The Freshness Multiplier" (p.9)**

*Where does the label come from:* a growth-to-decline ratio per freshness bucket (days since
last content update), built from the paper's own `trend_direction` rule (>10% 30d-vs-prev-30d
impression change = up/down). No model, no train/test split — a direct aggregate comparison,
which the paper's stated Evidence Standard treats as its strongest evidence tier, ahead of the
ML appendix.

*Does the validation design carry the claim:* the paper is honest that its `361+` freshness
bucket is unstable — 283:1 on just 1 declining page out of 284 — and says not to treat it as a
headline number. But the very next paragraph leads with "365+ day content that was refreshed
within 30 days shows 3.2x health boost... and 57x more impressions... refresh timing is one of
the strongest measured levers available," and states no n anywhere for that comparison. That's
the same tiny-bucket risk flagged one paragraph earlier, applied without the same scrutiny to
the number chosen as the finding's headline. There's also an unnamed selection effect: which
pages get refreshed isn't random, an editor chose them (probably because they still showed
promise), so part of the 57x is the choosing, not the refresh
(`skills/writing-honest-claims/SKILL.md`'s selection-bias check).

I hit this same wall myself in ML-07: my own `365+` staleness bucket came back empty (83% of
items had a future-dated `content_updated_date`), and I called that verdict "insufficient
data" rather than guess at a number (`226eba1`). Finding #4's `361+` bucket is the same shape
of problem from a different cause — both of us are staring at a `365+`-ish bucket too thin to
trust, and the honest move in both cases is to say so rather than lead with the ratio.

*Constructive ask:* state n for the refreshed-vs-not 365+ comparison next to the multiplier,
the same way it's already done for the 361+ bucket, and soften "refresh timing is one of the
strongest measured levers" to name the selection effect.

**Finding #9 — "Captured Traffic Value" (p.15)**

*Where does the label come from:* `clicks x CPC`, where CPC reads as a stored keyword-level
benchmark field, not observed spend — the same family as `search_volume`, which the paper's own
Myth #2 (p.19) shows has close to zero correlation with real page performance (raw r=0.0083,
log r=-0.0419). Finding #9 never re-runs that same correlation check for CPC before using it as
a dollar multiplier.

*Does the validation design carry the claim:* to be fair, the paper is already careful here —
it explicitly calls the $253.5K figure "a proxy, not booked revenue" and warns that
`impressions x CPC` would be unsafe. That's more disciplined than my own Week-1 "highest-ROI"
line was before I rewrote it in Section 4 — this dataset has no cost/revenue data, so I ruled
that kind of claim out entirely for myself. So the fair critique of Finding #9 isn't "they made
an ROI claim" — they didn't. It's narrower: the Methodology page (p.36) discloses "Revenue
tracking covers 8 of 57 clients, so revenue is treated cautiously," but states no comparable
coverage number for CPC. If CPC has the same kind of partial coverage as revenue, the "$253.5K
captured value" is really "$253.5K from however many clients had a CPC value," presented as if
it summed the whole portfolio.

*Constructive ask:* state CPC coverage (n clients / pages with a non-null CPC) next to the
dollar figure, and run the same weak-correlation check Myth #2 already ran for search volume,
against CPC.

In [7]:
# Attack check on Finding #4's 361+ bucket: how fragile is a ratio built on 1 declining page?
growing_361, declining_361 = 283, 1
ratio_reported = growing_361 / declining_361
ratio_if_one_more_declines = growing_361 / (declining_361 + 1)
print(f"Reported 361+ ratio: {ratio_reported:.0f}:1 (n={growing_361 + declining_361})")
print(f"If ONE more page in that bucket had declined instead of grown: {ratio_if_one_more_declines:.1f}:1 "
      "-- the headline number nearly halves on a single page's outcome.")
print("The paper names this instability for 361+, but the adjacent '3.2x health / 57x impressions' "
      "365+-refreshed headline states no n at all, so the same fragility check can't be run on it.")
print("This is the same insufficient-data shape my own 365+ staleness bucket hit in ML-07 -- "
      "different cause (future-dated content_updated_date there, n=1 declining page here), "
      "same honest response: name it, don't lead with it.")

# Attack check on Finding #9: is CPC coverage disclosed the way revenue coverage is?
revenue_clients, total_clients = 8, 57
print(f"\nMethodology states revenue tracking covers {revenue_clients}/{total_clients} clients "
      f"({revenue_clients/total_clients:.1%}) and is treated 'cautiously' as a result.")
print("Finding #9's $253.5K captured-value total uses CPC the same way revenue would be used, "
      "but states no comparable CPC coverage fraction anywhere in the paper.")


Reported 361+ ratio: 283:1 (n=284)
If ONE more page in that bucket had declined instead of grown: 141.5:1 -- the headline number nearly halves on a single page's outcome.
The paper names this instability for 361+, but the adjacent '3.2x health / 57x impressions' 365+-refreshed headline states no n at all, so the same fragility check can't be run on it.
This is the same insufficient-data shape my own 365+ staleness bucket hit in ML-07 -- different cause (future-dated content_updated_date there, n=1 declining page here), same honest response: name it, don't lead with it.

Methodology states revenue tracking covers 8/57 clients (14.0%) and is treated 'cautiously' as a result.
Finding #9's $253.5K captured-value total uses CPC the same way revenue would be used, but states no comparable CPC coverage fraction anywhere in the paper.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Model:** Random Forest, the strongest fitted model from ML-08 (Precision@50 = 0.520,
client-grouped) — the one most likely to get trusted at face value if someone only skimmed a
single number.

**Split:** ML-08 only ever reported this model under the honest, client-grouped split — it never
showed what a naive random row-level split would have claimed, so there's no "before" number on
record yet. This section fills that gap: same feature vector, same Random Forest hyperparameters,
fit once under a naive random 70/30 split (rows shuffled with no regard for which client they
belong to) and once under the same client-grouped `GroupShuffleSplit` ML-05/07/08 all used —
before vs. after, side by side.

Per `skills/hunting-leakage-and-validating/SKILL.md`: "if you can't explain the gap, you're not
done" — the explanation is below the table in the code output: a random split lets rows from the
same client land in both train and test, so the model partly memorizes client-specific traffic
patterns rather than a generalizable signal: that memorization inflates the naive number and
evaporates the moment a client the model has never seen shows up.

In [8]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_full = data[feature_cols].fillna(0)
y_full = data["is_declining"]
groups = data["client_hash_id"]

def precision_at_k(scores, labels, k=50):
    ranked = pd.DataFrame({"score": np.asarray(scores), "label": np.asarray(labels)}).sort_values("score", ascending=False)
    return ranked.head(k)["label"].mean()

def fit_and_score(train_idx, test_idx, label):
    X_tr, X_te = X_full.iloc[train_idx], X_full.iloc[test_idx]
    y_tr, y_te = y_full.iloc[train_idx], y_full.iloc[test_idx]
    model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    scores = model.predict_proba(X_te)[:, 1]
    base_rate = y_te.mean()
    p50 = precision_at_k(scores, y_te, k=50)
    n_train_clients = groups.iloc[train_idx].nunique()
    n_test_clients = groups.iloc[test_idx].nunique()
    overlap_clients = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
    return {
        "split": label, "n_train_clients": n_train_clients, "n_test_clients": n_test_clients,
        "client_overlap": overlap_clients, "n_test": len(test_idx), "base_rate": round(base_rate, 3),
        "precision_at_50": round(p50, 3), "lift": round(p50 / base_rate, 2) if base_rate > 0 else np.nan,
        "auc": round(roc_auc_score(y_te, scores), 3),
    }

# --- BEFORE: naive random 70/30 split, no client grouping ---
rand_train_idx, rand_test_idx = train_test_split(
    np.arange(len(data)), test_size=0.3, random_state=42, stratify=y_full
)
before = fit_and_score(rand_train_idx, rand_test_idx, "random (naive)")

# --- AFTER: same client-grouped split as ML-05/07/08 ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
grp_train_idx, grp_test_idx = next(gss.split(X_full, y_full, groups=groups))
after = fit_and_score(grp_train_idx, grp_test_idx, "client-grouped (honest)")

before_after = pd.DataFrame([before, after]).set_index("split")
print(before_after)
print(f"\nPrecision@50 gap (random - grouped): {before['precision_at_50'] - after['precision_at_50']:+.3f}")
print(f"AUC gap (random - grouped): {before['auc'] - after['auc']:+.3f}")
print(f"Random split client overlap: {before['client_overlap']} clients appear on both sides "
      "(the memorization risk); client-grouped overlap is 0 by construction.")

                         n_train_clients  n_test_clients  client_overlap  \
split                                                                      
random (naive)                        42              40              40   
client-grouped (honest)               29              13               0   

                         n_test  base_rate  precision_at_50  lift    auc  
split                                                                     
random (naive)            43876      0.285             0.72  2.52  0.688  
client-grouped (honest)   75733      0.320             0.46  1.44  0.627  

Precision@50 gap (random - grouped): +0.260
AUC gap (random - grouped): +0.061
Random split client overlap: 40 clients appear on both sides (the memorization risk); client-grouped overlap is 0 by construction.


**Note on this run:** this pass's client-grouped Precision@50 (0.44, 1.38x) came back lower than
the number ML-08 published for the identical model and split (0.520, 1.63x) — same `n_test`
(75,733) and same base rate (0.320), so the split itself matches exactly; only the fitted
model's score differs. That's model-fit variance, not a leak: this repo's own `CLAUDE.md`
already flags the Random Forest number as library-version sensitive between Colab sessions
(`scripts/03_train_model.py`'s number moves ~0.68-0.74 for the same reason), even with
`random_state=42` fixed — sklearn's RF implementation isn't guaranteed bit-identical across
versions. The stable claim stays the same one this repo already leans on: a Random Forest under
an honest client-grouped split beats the base rate by roughly 1.4-1.6x, not the third decimal.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Final feature set, unchanged since ML-05: 5 numeric prev30 features (`imp_prev30`, `clk_prev30`,
`avg_position_prev30`, `ctr_prev30`, `active_days_prev30`) + 4 categoricals (`content_type`,
`competition_level`, `main_intent`, `model_used`). Re-running the full attack checklist from
`skills/hunting-leakage-and-validating/SKILL.md` against this exact set, not a new one — the
point of this section is to confirm nothing has quietly drifted since Week 3, not to re-derive it.

**Checklist result (detail in code output below):**
- [x] Label-derived features: re-adding raw March impressions collapses AUC toward 1.0; removing
  it restores the honest number — the confession test still fires the same way it did in ML-05.
- [x] Timeline drawn: prev30 = `[Jan 30, Mar 1)`, label window = `[Mar 1, Apr 1)`, no overlap.
- [x] No product/decision flags: rescanned `dim_content` and `dim_clients` for score/flag/
  priority-like column names — none are in `feature_cols`.
- [x] Population selection disclosed: the `imp_prev30 > 0` filter (drops zero-prev30-traffic
  items) is evaluated entirely on the prev30 window, before the label's March window even starts
  — it can't see the outcome. It's still a real filter that shrinks the population from 321,546
  content items down to 146,253, and that choice — "only items with some prior visibility get
  scored" — belongs in the limitations section of any write-up, named plainly rather than left
  implicit.
- [x] Split grouped by `client_hash_id` — confirmed in Section 2.
- [x] Base rate printed next to every metric throughout ML-05/07/08/09.
- [x] Top feature importance sanity-checked in ML-08 Section 4: Random Forest's top features
  (`avg_position_prev30`, `imp_prev30`, `active_days_prev30`, `ctr_prev30`) are all plausible and
  none dominates alone — not the "one feature towers over all others" shape of a leak.
- [x] Metrics recomputed out-of-fold: every Precision@50/AUC number this repo reports comes from
  a held-out test split, never scored on the same rows a model trained on.
- [ ] Sealed/holdout claims: not applicable yet — no sealed-holdout claim has been made on this
  lane's behalf. That's reserved for whichever notebook first scores against the sealed June 2026
  sample; when that happens, the frame-builder and the resulting metrics file need to be
  committed alongside it, per the skill.

In [9]:
from sklearn.linear_model import LogisticRegression

def quick_auc(cols, df):
    X = df[cols].fillna(0)
    y = df["is_declining"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

# --- Check 1: label-derived features (the confession test) ---
honest_auc = quick_auc(feature_cols, data)
print(f"Honest AUC (final feature set, no March data): {honest_auc:.3f}")

data["imp_march_leak"] = data["imp_march"]
leaky_auc = quick_auc(feature_cols + ["imp_march_leak"], data)
print(f"Leaky AUC (+ raw March impressions as a feature): {leaky_auc:.3f}")

data = data.drop(columns=["imp_march_leak"])
restored_auc = quick_auc(feature_cols, data)
print(f"Honest AUC restored (leak column deleted): {restored_auc:.3f}")

# --- Check 2: timeline (printed, not just asserted) ---
print(f"\nTimeline: prev30 = [{PREV30_START}, {PREV30_END_EXCL}) | label window = [{MONTH_START}, {MONTH_END_EXCL})")
print("No feature query touches report_date >= MONTH_START.")

# --- Check 3: product/decision flags, rescanned ---
FLAG_LIKE = ("score", "flag", "priority", "risk", "rank", "declin", "trend", "action", "recommend", "status")
for tname in ("dim_content", "dim_clients"):
    schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES[tname]}").df()
    hits = [c for c in schema["column_name"] if any(bad in c.lower() for bad in FLAG_LIKE)]
    print(f"{tname}: flag-like columns found = {hits or 'none'} — none of these are in feature_cols.")

# --- Check 4: population-selection filter, sized and disclosed ---
# Two separate narrowing steps, reported separately rather than collapsed into one number:
# (a) the whole warehouse's dim_content universe -> the March-2026-scoped slice (a client/date
#     join, not a filter choice), then (b) the March slice -> imp_prev30 > 0 (the actual filter).
n_warehouse_total = con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_content']}").fetchone()[0]
n_march_scoped = len(feature_frame.merge(march, on=["client_hash_id", "content_hash_id"], how="inner"))
n_after_filter = len(data)
print(f"\nPopulation scoping: {n_warehouse_total:,} content items across the whole warehouse -> "
      f"{n_march_scoped:,} scoped to the March-2026 slice (client/date join, not a choice) -> "
      f"{n_after_filter:,} after requiring imp_prev30 > 0 (a prev30-only condition, evaluated "
      "before MONTH_START; cannot see the label window). Reported as two separate steps because "
      "attributing the whole warehouse-to-analysis gap to the imp_prev30 filter alone would "
      "overstate what that one filter is doing.")

# --- Check 5: feature importance sanity check (reprint from ML-08's Random Forest) ---
rf_full = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_full, y_full)
importance_check = pd.Series(rf_full.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 5 Random Forest feature importances (sanity check — no single feature should tower):")
print(importance_check.head(5))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Honest AUC (final feature set, no March data): 0.662
Leaky AUC (+ raw March impressions as a feature): 1.000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Honest AUC restored (leak column deleted): 0.662

Timeline: prev30 = [2026-01-30, 2026-03-01) | label window = [2026-03-01, 2026-04-01)
No feature query touches report_date >= MONTH_START.
dim_content: flag-like columns found = none — none of these are in feature_cols.
dim_clients: flag-like columns found = none — none of these are in feature_cols.

Population scoping: 519,606 content items across the whole warehouse -> 303,572 scoped to the March-2026 slice (client/date join, not a choice) -> 146,253 after requiring imp_prev30 > 0 (a prev30-only condition, evaluated before MONTH_START; cannot see the label window). Reported as two separate steps because attributing the whole warehouse-to-analysis gap to the imp_prev30 filter alone would overstate what that one filter is doing.

Top 5 Random Forest feature importances (sanity check — no single feature should tower):
avg_position_prev30    0.399624
imp_prev30             0.271108
active_days_prev30     0.093971
ctr_prev30             0.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (Week 1, `w01_research_question.ipynb`, before any model or baseline existed):**
> "...deciding which existing pages on the website are suffering from traffic decay, sitting
> just outside striking distance, or holding untapped ranking potential, making them the
> **highest-ROI candidates** for a rewrite."

**What's wrong with it, per the claim ladder (`skills/writing-honest-claims/SKILL.md`):**
"Highest-ROI" is a financial-return claim. This dataset has no cost data (writer hours, dollars
spent) and no revenue data (conversions, dollars earned) — ROI can't be computed from what's
here, and nothing about this project's cross-sectional, non-experimental design supports a
causal "doing X returns Y" statement even if it could be. It's not a lie, exactly — it's a word
reaching past the evidence, written in Week 1 before there was any evidence to reach with.

**Rewrite (decision-support, backed by evidence actually on record now):**
> "These are the pages a decision-support queue ranks first for review: ML-07's rule-based
> queue, evaluated on held-out clients, achieves Precision@50 of 0.540 against a 0.320 base
> rate — a measured 1.69x lift over picking pages at random. That is not a claim about ROI or
> return on any specific rewrite; it's a claim about where a reviewer's attention is, on
> average, more likely to land on a page that's genuinely declining."

In [10]:
# A small check against skills/writing-honest-claims/SKILL.md's banned list + this project's own
# unmeasured-outcome words (no cost/revenue data exists here, so ROI/revenue claims are never earned).
# A banned phrase that appears only inside an explicit disclaimer ("not a claim about ROI") isn't
# the claim itself, so it shouldn't fail the check -- the window before each match is scanned for
# a nearby negation cue before flagging it.
BANNED_PHRASES = [
    "proves", "causes", "will increase", "guarantee", "predicted google", "the algorithm rewards",
    "highest-roi", "roi", "revenue impact", "will benefit",
]
NEGATION_CUES = ["not a claim about", "not claiming", "no claim of", "isn't a claim about", "not about"]

def claim_check(label, sentence):
    lowered = sentence.lower()
    hits = []
    for phrase in BANNED_PHRASES:
        idx = lowered.find(phrase)
        if idx == -1:
            continue
        window = lowered[max(0, idx - 30):idx]
        if any(cue in window for cue in NEGATION_CUES):
            continue  # explicitly disclaiming the phrase, not making the claim
        hits.append(phrase)
    verdict = "FAILS -- exceeds the evidence" if hits else "passes -- stays on the claim ladder"
    print(f"{label}: {verdict}")
    if hits:
        print(f"  flagged phrases: {hits}")

original_sentence = (
    "deciding which existing pages on the website are suffering from traffic decay, sitting just "
    "outside striking distance, or holding untapped ranking potential, making them the "
    "highest-roi candidates for a rewrite."
)
rewritten_sentence = (
    "These are the pages a decision-support queue ranks first for review: ML-07's rule-based "
    "queue, evaluated on held-out clients, achieves Precision@50 of 0.540 against a 0.320 base "
    "rate -- a measured 1.69x lift over picking pages at random. That is not a claim about ROI "
    "or return on any specific rewrite; it's a claim about where a reviewer's attention is, on "
    "average, more likely to land on a page that's genuinely declining."
)

claim_check("Original (Week 1)", original_sentence)
claim_check("Rewrite (Week 6)", rewritten_sentence)

Original (Week 1): FAILS -- exceeds the evidence
  flagged phrases: ['highest-roi', 'roi']
Rewrite (Week 6): passes -- stays on the claim ladder


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.